<div style="text-align: center;">
  <h1 style="text-align: center;">Reto 2 — Asistente de Guías de Viaje</h1>
  <h3 style="text-align: center;">Máster en Data Science e IA · EBIS Business Techschool · Bloque B</h3>
  <p style="text-align: center;">Workshop: Prototipado de Agentes</p>
</div>

## Qué construimos

- Ingesta de 3 PDFs de guías de viaje (Japón, Tailandia, Portugal) en Chroma
- Tools: `search_kb` (búsqueda semántica) + `fetch_section` (lectura directa de PDF)
- Salida tipada: `Respuesta(texto, citas, confianza)`
- Guardrail de salida: exige al menos una cita real
- Guardrail de entrada: clasificador LLM que rechaza preguntas off-topic
- Eval set de 5 preguntas con citation rate

> ⚠️ **Las guías son ficticias**, creadas para este ejercicio: algunos datos no son reales (por ejemplo, los requisitos de entrada a Japón). No las uses para viajar. Y es una lección en sí misma: **una respuesta puede estar bien citada y ser falsa** si la fuente está mal o desactualizada. Citar demuestra de dónde sale el dato, no que sea verdad.

### 🧭 Antes de empezar

- **Cómo se usa:** ejecuta las celdas en orden, de arriba abajo, con `Mayús + Enter`. Si una celda da error, lee la última línea: casi siempre dice qué falta.
- **Si algo se queda raro:** menú *Entorno de ejecución → Reiniciar y ejecutar todo*.
- **Tu API key:** en Colab, guárdala como secreto `OPENAI_API_KEY` (icono de la llave, a la izquierda). Nunca la pegues dentro de una celda.

**Vocabulario de este notebook**

| Término | Qué es |
|---|---|
| **RAG** | buscar en las guías de viaje y responder con lo encontrado, citando fichero y página. |
| **Chroma** | la base de datos vectorial donde se guardan los trozos de las guías. |
| **`search_kb`** | la herramienta del agente que busca en Chroma. |
| **Guardrail** | el filtro que rechaza respuestas sin citas o preguntas que no son de viajes. |
| **Eval set** | las 5 preguntas de `tests/cases.yaml` con las que se comprueba el agente. |


## Setup

In [ ]:
!pip install -q --upgrade "openai-agents>=0.24" nest_asyncio pandas chromadb pypdf sentence-transformers pyyaml
# ⚠️ Tras instalar/actualizar paquetes, REINICIA el kernel (Kernel → Restart Kernel).

In [ ]:
import os
from getpass import getpass

# La clave NUNCA se escribe en el notebook: secretos de Colab -> variable de entorno -> teclado
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")
os.environ["OPENAI_DEFAULT_MODEL"] = "gpt-6-luna"   # modelo por defecto de todos los agentes

In [ ]:
import nest_asyncio
nest_asyncio.apply()  # permite Runner.run_sync dentro de Jupyter

import os
from pathlib import Path

MANUALS_DIR = Path("data/manuals")
CHROMA_DIR = Path("data/chroma")
EVAL_FILE = Path("tests/cases.yaml")

MANUALS_DIR.mkdir(parents=True, exist_ok=True)
CHROMA_DIR.mkdir(parents=True, exist_ok=True)
EVAL_FILE.parent.mkdir(parents=True, exist_ok=True)


# En Colab las guías y el eval set no están: se descargan del repositorio del curso si faltan
import urllib.request
REPO = "https://raw.githubusercontent.com/alexoli97/MasterEBIS/main/S4_Agentes_Avanzado/OpenAI_Agents/"
for ruta in ["data/manuals/guia_japon.pdf", "data/manuals/guia_portugal.pdf",
             "data/manuals/guia_tailandia.pdf", "tests/cases.yaml"]:
    if not Path(ruta).exists():
        urllib.request.urlretrieve(REPO + ruta, ruta)
        print("Descargado:", ruta)

print("✅ Setup completado.")
print("Guías en:", MANUALS_DIR.resolve())
print("Ficheros:", [p.name for p in MANUALS_DIR.glob("*.pdf")])

## TODO 1 — Ingesta de guías en Chroma ✅

Una entrada por página. Metadatos `{manual, pagina}`. ID determinista para que `upsert` sea idempotente.

In [ ]:
import logging
logging.getLogger("chromadb.telemetry.product.posthog").setLevel(logging.CRITICAL)

import chromadb
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

# Modelo local — se descarga una vez (~90 MB) y luego va en CPU sin red
_modelo = SentenceTransformer("all-MiniLM-L6-v2")

def _embed(textos: list[str]) -> list[list[float]]:
    # Sin llamadas HTTP — corre en local en milisegundos
    return _modelo.encode(textos, show_progress_bar=False).tolist()

client_chroma = chromadb.Client(settings=chromadb.Settings(anonymized_telemetry=False))
coleccion = client_chroma.get_or_create_collection("guias_viaje")

def ingestar_guias(directorio: Path, force: bool = False) -> int:
    """
    TODO 1 — Lee todos los PDFs de `directorio` y los sube a Chroma.

    Pasos:
      1. Itera sobre cada PDF con `sorted(directorio.glob("*.pdf"))`.
      2. Por cada página extrae el texto con PdfReader y sáltate las vacías.
      3. Acumula: docs (texto), metas ({"manual": pdf.name, "pagina": i}), ids (f"{pdf.name}::{i}").
      4. Calcula todos los embeddings de golpe: `embeddings = _embed(docs)`.
      5. Sube todo con: `coleccion.upsert(documents=docs, embeddings=embeddings, metadatas=metas, ids=ids)`.
      6. Devuelve `len(docs)`.

    El id determinista hace que re-ejecutar la celda sea seguro (upsert = insert or update).
    """
    if coleccion.count() > 0 and not force:
        print(f"⏩  Colección ya tiene {coleccion.count()} docs. Skip (pasa force=True para re-ingestar).")
        return coleccion.count()
    docs, metas, ids = [], [], []
    for pdf in sorted(directorio.glob("*.pdf")):
        reader = PdfReader(str(pdf))
        for i, page in enumerate(reader.pages, start=1):
            texto = (page.extract_text() or "").strip()
            if not texto:
                continue
            docs.append(texto)
            metas.append({"manual": pdf.name, "pagina": i})
            ids.append(f"{pdf.name}::{i}")
    if docs:
        print(f"  → Calculando embeddings para {len(docs)} páginas en local...")
        embeddings = _embed(docs)
        coleccion.upsert(documents=docs, embeddings=embeddings, metadatas=metas, ids=ids)
    return len(docs)

total = ingestar_guias(MANUALS_DIR)
print(f"✅ Ingestadas {total} páginas")

## TODO 2 — Tool `search_kb` ✅

In [ ]:
from agents import function_tool
from pydantic import BaseModel

class Fragmento(BaseModel):
    texto: str
    manual: str
    pagina: int

@function_tool
def search_kb(query: str, k: int = 4) -> list[Fragmento]:
    """
    TODO 2 — Busca semánticamente en Chroma y devuelve los k fragmentos más relevantes.

    Pasos:
      1. Embede la query: `query_embedding = _embed([query])[0]`.
      2. Consulta Chroma: `res = coleccion.query(query_embeddings=[query_embedding], n_results=k)`.
      3. Convierte cada resultado en un `Fragmento(texto=..., manual=..., pagina=...)`.
         Los resultados están en `res["documents"][0]` y `res["metadatas"][0]`.

    Args:
        query: Pregunta del viajero en lenguaje natural.
        k: Número de fragmentos a devolver (por defecto 4).
    """
    print(f"  🔎 search_kb({query[:60]!r}, k={k})")
    query_embedding = _embed([query])[0]
    res = coleccion.query(query_embeddings=[query_embedding], n_results=k)
    return [
        Fragmento(texto=doc, manual=meta["manual"], pagina=int(meta["pagina"]))
        for doc, meta in zip(res["documents"][0], res["metadatas"][0])
    ]

## TODO 3 — Tool `fetch_section` ✅

In [ ]:
@function_tool
def fetch_section(manual: str, pagina: int) -> str:
    """
    TODO 3 — Lee una página completa de un PDF y devuelve su texto.

    Pasos:
      1. Construye la ruta: `path = MANUALS_DIR / manual`.
      2. Si no existe, devuelve un string de error (no lances excepción).
      3. Abre con `PdfReader(str(path))` y accede a `reader.pages[pagina - 1]`.
      4. Devuelve `page.extract_text()` o "[página vacía]" si está en blanco.
      5. Valida que `pagina` esté en rango (1 a len(reader.pages)).

    Args:
        manual: Nombre del fichero PDF (ej: 'guia_japon.pdf').
        pagina: Número de página empezando en 1.
    """
    print(f"  📄 fetch_section({manual}, p{pagina})")
    path = MANUALS_DIR / manual
    if not path.exists():
        return f"[error] Guía no encontrada: {manual}"
    reader = PdfReader(str(path))
    if pagina < 1 or pagina > len(reader.pages):
        return f"[error] Página {pagina} fuera de rango (1-{len(reader.pages)})"
    return reader.pages[pagina - 1].extract_text() or "[página vacía]"

## TODO 4 — Salida tipada + Guardrails ✅

In [ ]:
from agents import (
    Agent, Runner, output_guardrail, input_guardrail,
    GuardrailFunctionOutput, RunContextWrapper,
)
from pydantic import BaseModel, Field

class Cita(BaseModel):
    manual: str
    pagina: int

class Respuesta(BaseModel):
    texto: str = Field(description="Respuesta en lenguaje natural, 3-5 frases")
    citas: list[Cita] = Field(description="Al menos una cita a una guía de viaje")
    confianza: float = Field(description="Confianza 0-1 basada en la calidad de los fragmentos")

@output_guardrail
async def exigir_citas(
    ctx: RunContextWrapper[None],
    agent: Agent,
    output: Respuesta,
) -> GuardrailFunctionOutput:
    """
    TODO 4 — Guardrail de salida: rechaza respuestas sin citas válidas.

    Pasos:
      1. Filtra `output.citas` quedándote solo con las que tienen `manual` no vacío y `pagina > 0`.
      2. Devuelve `GuardrailFunctionOutput(output_info="...", tripwire_triggered=...)`:
         `True` si la lista filtrada está vacía (bloquea), `False` si hay citas (deja pasar).

    Tip: un guardrail NO lanza excepciones; si `tripwire_triggered=True`, el SDK corta y lanza `OutputGuardrailTripwireTriggered`.
    """
    citas_validas = [c for c in output.citas if c.manual and c.pagina > 0]
    return GuardrailFunctionOutput(output_info=f"{len(citas_validas)} citas válidas", tripwire_triggered=not citas_validas)

# BONUS — Input guardrail con clasificador LLM (más robusto que una lista de palabras clave):

class _TopicCheck(BaseModel):
    es_viajes: bool
    motivo: str

_clasificador = Agent(
    name="Clasificador de tópico",
    instructions="¿Es esta pregunta sobre viajes, turismo o planificación de viajes? "
                 "Devuelve es_viajes=true aunque la pregunta sea vaga.",
    model="gpt-6-luna",
    output_type=_TopicCheck,
)

@input_guardrail
async def rechazar_offtopic(
    ctx: RunContextWrapper[None],
    agent: Agent,
    input_text: str,
) -> GuardrailFunctionOutput:
    res = await Runner.run(_clasificador, input_text)
    check = res.final_output
    return GuardrailFunctionOutput(output_info=check.motivo, tripwire_triggered=not check.es_viajes)

## TODO 5 — Construir el agente ✅

In [ ]:
SYSTEM_PROMPT = """
Eres TravelMind, el asistente de viajes de EBIS Business Techschool.
Tu única fuente de verdad son las guías de viaje internas accesibles vía las tools.
Reglas:
1. Antes de responder, llama a `search_kb` con la pregunta del viajero.
2. Si necesitas más contexto, usa `fetch_section` para leer la página completa.
3. Responde en 3-5 frases. Sé concreto y accionable.
4. Incluye SIEMPRE al menos una cita (guía + página). Si no hay fuente, dilo.
5. Si la pregunta no está cubierta por las guías, indícalo claramente.
""".strip()

agente_guias = Agent(
    name="TravelMind Guías",
    instructions=SYSTEM_PROMPT,
    tools=[search_kb, fetch_section],
    output_type=Respuesta,
    output_guardrails=[exigir_citas],
    input_guardrails=[rechazar_offtopic],
)
print("✅ Agente listo")

## TODO 6 — Probar el agente ✅

In [ ]:
from agents import Runner, trace

PREGUNTA = "¿Cuántos días puede quedarse un español en Tailandia sin visado?"

with trace("reto2 prueba manual") as t:
    resultado = Runner.run_sync(agente_guias, PREGUNTA)
    trace_id = t.trace_id

r = resultado.final_output
print("📝", r.texto)
print()
print("📑 Citas:")
for c in r.citas:
    print(f"   - {c.manual} · p{c.pagina}")
print(f"\n🔎 Confianza: {r.confianza}")
print(f"\n🔗 Traza: https://platform.openai.com/traces/trace?trace_id={trace_id}")

## TODO 7 — Eval set ✅

In [ ]:
import time
import yaml

def correr_eval():
    """
    TODO 7 — Corre el agente contra los 5 casos de `tests/cases.yaml` y mide resultados.

    Cada caso tiene: id, pregunta, keyword, manual_esperado.

    Pasos:
      1. Carga los casos: `casos = yaml.safe_load(EVAL_FILE.read_text())`.
      2. Para cada caso, llama a `Runner.run_sync(agente_guias, caso["pregunta"])`.
      3. Comprueba que `caso["keyword"].lower()` aparece en `r.texto.lower()`.
      4. Comprueba que `r.citas` no está vacío.
      5. Comprueba que alguna cita apunta a `caso["manual_esperado"]`. El caso pasa solo si se cumplen 3, 4 y 5.
      6. Imprime ✅/❌ por caso y el resumen final: pasados/total + citation rate.
      7. (Bonus) Para los casos con `sin_respuesta`, aprueba si salta `OutputGuardrailTripwireTriggered`
         o si el texto dice que la información no está en las guías.
      8. (Extra) Comprueba que la PÁGINA citada contiene el dato: lee esa página con PdfReader y busca la keyword.
         Una cita a la guía correcta pero a una página que no lo dice tampoco respalda la respuesta.
    """
    from agents.exceptions import OutputGuardrailTripwireTriggered

    todos = yaml.safe_load(EVAL_FILE.read_text())
    casos = [c for c in todos if not c.get("sin_respuesta")]
    pasados = 0
    citas_correctas = 0
    latencias = []
    for caso in casos:
        t0 = time.perf_counter()
        try:
            res = Runner.run_sync(agente_guias, caso["pregunta"])
            r = res.final_output
        except Exception as e:
            print(f"❌ {caso['id']}: error {type(e).__name__}: {e}")
            continue
        lat = time.perf_counter() - t0
        latencias.append(lat)

        keyword_ok = caso["keyword"].lower() in r.texto.lower()
        cita_ok = bool(r.citas)
        manual_ok = any(
            c.manual == caso.get("manual_esperado") for c in r.citas
        ) if caso.get("manual_esperado") else cita_ok

        ok = keyword_ok and cita_ok and manual_ok   # el dato y la guía correcta: una cita cualquiera no vale
        # ¿la página citada contiene el dato? (citar la guía buena en una página que no lo dice no respalda nada)
        pagina_ok = any(c.manual == caso.get("manual_esperado") and 0 < c.pagina <= len(PdfReader(MANUALS_DIR / c.manual).pages)
                        and caso["keyword"].lower() in (PdfReader(MANUALS_DIR / c.manual).pages[c.pagina - 1].extract_text() or "").lower()
                        for c in r.citas)
        if ok:
            pasados += 1
        if manual_ok and pagina_ok:
            citas_correctas += 1

        status = "✅" if ok else "❌"
        print(f"{status} {caso['id']} ({lat:.1f}s)")
        print(f"    keyword '{caso['keyword']}' {'sí' if keyword_ok else 'NO'} · "
              f"citas {len(r.citas)} · manual {'OK' if manual_ok else 'mal'} · página {'respalda' if pagina_ok else 'NO respalda'}")
        if r.citas:
            print(f"    → {r.citas[0].manual} p{r.citas[0].pagina}")
    # Bonus: preguntas que NO están en las guías. Aprueba si el agente no inventa.
    NO_SE = ("no aparece", "no está", "no encuentro", "no tengo", "no cubre", "no incluye", "no figura", "no dispongo",
             "no contiene", "no puedo", "no hay información")
    for caso in [c for c in todos if c.get("sin_respuesta")]:
        try:
            r = Runner.run_sync(agente_guias, caso["pregunta"]).final_output
            ok = any(x in r.texto.lower() for x in NO_SE)
            detalle = r.texto[:120]
        except OutputGuardrailTripwireTriggered:
            ok, detalle = True, "el guardrail cortó la respuesta (sin citas): no se inventó nada, pero revisa el motivo"
        print(f"{'⭐' if ok else '❌'} bonus {caso['id']}: {detalle}")

    n = len(casos)
    print(f"\n📊 Pasados: {pasados}/{n}  (objetivo: {max(1, round(0.8 * n))})")
    print(f"📑 Citation rate: {citas_correctas}/{n}")
    if latencias:
        print(f"⏱  Latencia media: {sum(latencias)/len(latencias):.1f}s")

correr_eval()